In [ ]:
# =========================
# 0. Import Libraries
# =========================
import pandas as pd
import numpy as np
from google.colab import drive
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from scipy import stats
from datetime import datetime
import csv
import re
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.ticker import FuncFormatter
from matplotlib.ticker import ScalarFormatter
import os
import glob

In [ ]:
# =========================
# 1. Load Data
# =========================

# Mount Google Drive
drive.mount('/content/drive')


# Lokasi folder data transaksi
folder_transaksi = "data/transaksi/"

# Lokasi file data customer
file_customer = "data/customer.csv"


# =========================
# 1A. Ambil Semua File Transaksi
# =========================

list_file_transaksi = glob.glob(
    os.path.join(folder_transaksi, '*.csv')
)

if len(list_file_transaksi) == 0:
    raise FileNotFoundError(
        f'Tidak ada file CSV di folder: {folder_transaksi}'
    )


# Fungsi untuk mengambil nomor transaksi dari nama file
def ambil_nomor_transaksi(file_path):
    nama_file = os.path.basename(file_path)

    hasil = re.search(
        r'Transaksi(\d+)',
        nama_file,
        re.IGNORECASE
    )

    if hasil:
        return int(hasil.group(1))

    return 0


# Urutkan file dari Transaksi12 sampai Transaksi01
list_file_transaksi = sorted(
    list_file_transaksi,
    key=ambil_nomor_transaksi,
    reverse=True
)


# =========================
# 1B. Baca dan Gabungkan Data Transaksi
# =========================

list_dataframe_transaksi = []

for file in list_file_transaksi:

    df_bulanan = pd.read_csv(
        file,
        encoding='utf-8-sig',
        low_memory=False
    )

    # Menambahkan nama file sumber
    df_bulanan['Sumber File'] = os.path.basename(file)

    list_dataframe_transaksi.append(df_bulanan)


# Gabungkan seluruh data transaksi
df_data = pd.concat(
    list_dataframe_transaksi,
    ignore_index=True
)


# =========================
# 1C. Urutkan Transaksi Berdasarkan Tanggal
# =========================

# Ubah Date menjadi datetime
df_data['Date'] = pd.to_datetime(
    df_data['Date'],
    format='%d-%m-%Y',
    errors='coerce'
)

# Urutkan dari tanggal terbaru ke tanggal terlama
df_data = df_data.sort_values(
    by='Date',
    ascending=False
).reset_index(drop=True)


# =========================
# 1D. Deteksi Delimiter Data Customer
# =========================

with open(
    file_customer,
    'r',
    encoding='utf-8-sig',
    errors='replace'
) as file:

    sample_customer = file.read(10000)


try:
    delimiter_customer = csv.Sniffer().sniff(
        sample_customer,
        delimiters=[',', ';', '\t', '|']
    ).delimiter

except csv.Error:
    delimiter_customer = ','


# =========================
# 1E. Baca Data Customer
# =========================

rows_customer = []

with open(
    file_customer,
    'r',
    encoding='utf-8-sig',
    errors='replace',
    newline=''
) as file:

    reader = csv.reader(
        file,
        delimiter=delimiter_customer
    )

    # Ambil header pertama
    header_customer = next(reader)

    # Hilangkan spasi pada nama kolom
    header_customer = [
        str(column).strip()
        for column in header_customer
    ]

    jumlah_kolom_customer = len(header_customer)

    for row in reader:

        # Lewati baris kosong
        if not row:
            continue

        # Lewati baris yang seluruh isinya kosong
        if all(str(value).strip() == '' for value in row):
            continue

        # Lewati header yang muncul kembali di tengah file
        if str(row[0]).strip().lower() == 'name':
            continue

        # Jika jumlah kolom lebih banyak, ambil sesuai jumlah header
        if len(row) > jumlah_kolom_customer:
            row = row[:jumlah_kolom_customer]

        # Jika jumlah kolom kurang, tambahkan nilai kosong
        elif len(row) < jumlah_kolom_customer:
            row = row + [''] * (
                jumlah_kolom_customer - len(row)
            )

        rows_customer.append(row)


# Buat DataFrame customer
df_customer = pd.DataFrame(
    rows_customer,
    columns=header_customer
)

# Rapikan index
df_customer = df_customer.reset_index(drop=True)


# =========================
# 1F. Informasi Awal Data
# =========================

print('=== DATA BERHASIL DIMUAT ===')

print('\nData transaksi')
print('Jumlah file:', len(list_file_transaksi))
print('Jumlah baris:', df_data.shape[0])
print('Jumlah kolom:', df_data.shape[1])

print('\nData customer')
print('Jumlah baris:', df_customer.shape[0])
print('Jumlah kolom:', df_customer.shape[1])


print('\n=== KOLOM DATA TRANSAKSI ===')
print(df_data.columns.tolist())


print('\n=== KOLOM DATA CUSTOMER ===')
print(df_customer.columns.tolist())


print('\n=== MISSING VALUE DATA TRANSAKSI ===')
print(df_data.isnull().sum())


print('\n=== MISSING VALUE DATA CUSTOMER ===')
print(df_customer.isnull().sum())


print('\n=== PREVIEW DATA TRANSAKSI ===')
print(df_data.head())


print('\n=== PREVIEW DATA CUSTOMER ===')
print(df_customer.head())

In [ ]:
# # Install library
# !pip install --quiet gspread gspread_dataframe

# # Authentication
# from google.colab import auth
# auth.authenticate_user()

# import gspread
# from google.auth import default
# from gspread_dataframe import set_with_dataframe

# creds, _ = default()
# gc = gspread.authorize(creds)

# # Buat spreadsheet baru
# spreadsheet = gc.create("Data Transaksi Lintang Bakso")

# # Export DataFrame ke sheet pertama
# worksheet = spreadsheet.sheet1
# worksheet.update_title("Data Transaksi")
# set_with_dataframe(worksheet, df_data)

# # Tampilkan link spreadsheet
# print("Export berhasil!")
# print(spreadsheet.url)

In [ ]:
# =========================
# 2. Data Cleaning
# =========================


# Fungsi untuk memperbaiki pola angka 2
# Contoh:
# wi2n -> wiwin
# yo2k -> yoyok
# li2s -> lilis
def expand_2_pattern(text):
    text = str(text)

    while '2' in text:
        new_text = re.sub(r'([A-Za-z]{2})2([A-Za-z])', r'\1\1\2', text)

        if new_text == text:
            break

        text = new_text

    text = text.replace('2', '')
    return text


# Fungsi untuk standarisasi nama customer
def clean_customer_name(text):
    text = str(text).strip()
    text = re.sub(r'\s+', ' ', text)
    text = text.replace('.', '')

    # Ubah pola angka 2 jadi huruf ganda
    text = expand_2_pattern(text)

    # Hapus sapaan / prefix
    text = re.sub(r'(?i)\b(bpk|bapak|bp|pak|pk|pzk)\b', '', text)
    text = re.sub(r'(?i)\b(bu|ibu|ibuk|bok)\b', '', text)
    text = re.sub(r'(?i)\b(mbak|mbk|mas|maz|mbzk)\b', '', text)

    # Rapikan kembali spasi setelah sapaan dihapus
    text = re.sub(r'\s+', ' ', text).strip()

    # Kapitalisasi awal setiap kata
    text = text.title()

    return text


# Rapikan data customer: buang kolom Unnamed jika ada
df_customer = df_customer.loc[
    :,
    ~df_customer.columns.str.contains('^Unnamed')
].copy()


# Ambil dan samakan nama kolom data customer
if 'Name' in df_customer.columns and 'Phone' in df_customer.columns:

    df_customer = df_customer[['Name', 'Phone']].copy()

    df_customer = df_customer.rename(columns={
        'Name': 'Customer Name',
        'Phone': 'Customer Phone'
    })

else:

    df_customer = df_customer[
        ['Customer Name', 'Customer Phone']
    ].copy()


# Salin data transaksi dan samakan nama kolom Customer
if 'Customer' in df_data.columns:

    df_cleaned = df_data.rename(columns={
        'Customer': 'Customer Phone'
    }).copy()

else:

    df_cleaned = df_data.copy()


# Hapus baris data yang kolom Customer Phone kosong
df_cleaned = df_cleaned.dropna(
    subset=['Customer Phone']
).copy()


# Hapus baris customer yang kolom Customer Phone kosong
df_customer_cleaned = df_customer.dropna(
    subset=['Customer Phone']
).copy()


# Bersihkan Customer Phone di data transaksi
df_cleaned['Customer Phone'] = (
    df_cleaned['Customer Phone']
    .astype(str)
    .str.replace(r'\D', '', regex=True)
    .str.lstrip('0')
)


# Standarisasi awalan Indonesia pada data transaksi
df_cleaned['Customer Phone'] = df_cleaned['Customer Phone'].apply(
    lambda x: '62' + x if not x.startswith('62') else x
)


# Bersihkan Customer Phone di data customer
df_customer_cleaned['Customer Phone'] = (
    df_customer_cleaned['Customer Phone']
    .astype(str)
    .str.replace(r'\D', '', regex=True)
    .str.lstrip('0')
)


# Standarisasi awalan Indonesia pada data customer
df_customer_cleaned['Customer Phone'] = df_customer_cleaned['Customer Phone'].apply(
    lambda x: '62' + x if not x.startswith('62') else x
)


# Bersihkan Customer Name
df_customer_cleaned['Customer Name'] = (
    df_customer_cleaned['Customer Name']
    .apply(clean_customer_name)
)


# Hapus duplikat nomor HP di data customer
df_customer_cleaned = df_customer_cleaned.drop_duplicates(
    subset='Customer Phone',
    keep='first'
)


# Pastikan Net Sales bertipe numerik
df_cleaned['Net Sales'] = pd.to_numeric(
    df_cleaned['Net Sales'],
    errors='coerce'
)


# Hapus data Net Sales yang gagal dibaca
df_cleaned = df_cleaned.dropna(
    subset=['Net Sales']
).copy()


# Gabungkan nama customer berdasarkan Customer Phone
df_cleaned = df_cleaned.merge(
    df_customer_cleaned[
        ['Customer Phone', 'Customer Name']
    ],
    on='Customer Phone',
    how='left'
)


# Isi nama customer yang tidak ditemukan
df_cleaned['Customer Name'] = (
    df_cleaned['Customer Name']
    .fillna('Nama tidak ditemukan')
)


# Menampilkan kondisi setelah data cleaning
print("\n=== SETELAH DATA CLEANING ===")
print("Jumlah baris data:", df_cleaned.shape[0])

print(
    "Baris tanpa Customer Phone yang terhapus:",
    df_data.shape[0] - df_cleaned.shape[0]
)

print(
    "Jumlah baris tanpa nama customer:",
    (df_cleaned['Customer Name'] == 'Nama tidak ditemukan').sum()
)


print("\nPreview hasil merge:")
print(
    df_cleaned[
        [
            'Customer Phone',
            'Customer Name',
            'Receipt Number',
            'Date',
            'Net Sales'
        ]
    ].head()
)

In [ ]:
# =========================
# 3. Data Transformation (Recency, Frequency, Monetary)
# =========================

# Mengubah kolom Date menjadi tipe datetime
df_cleaned['Date'] = pd.to_datetime(
    df_cleaned['Date'],
    errors='coerce'
)

# Pastikan Net Sales bertipe numerik
df_cleaned['Net Sales'] = pd.to_numeric(
    df_cleaned['Net Sales'],
    errors='coerce'
)

# Hapus data yang tanggal atau nominalnya gagal dibaca
df_cleaned = df_cleaned.dropna(
    subset=['Date', 'Net Sales']
).copy()

# Tanggal terakhir transaksi sebagai acuan Recency
latest_date = df_cleaned['Date'].max()

# Menghitung RFM per customer
df_rfm_final = df_cleaned.groupby(
    ['Customer Phone', 'Customer Name']
).agg(
    Recency=('Date', lambda x: (latest_date - x.max()).days),
    Frequency=('Receipt Number', 'nunique'),
    Monetary=('Net Sales', 'sum')
).reset_index()

# Menampilkan data transformasi
print("\n=== SETELAH DATA TRANSFORMATION ===")
print(df_rfm_final.head())

In [ ]:
# =========================
# 4. Data Selection (Recency, Frequency, Monetary)
# =========================

# Mengambil kolom yang relevan untuk RFM
df_rfm_final = df_rfm_final[
    [
        'Customer Phone',
        'Customer Name',
        'Recency',
        'Frequency',
        'Monetary'
    ]
].copy()

# Lokasi penyimpanan data hasil preprocessing
file_output = (
    '/content/drive/My Drive/TA/1 Tahun Lintang Bakso/'
    'data-prepared.csv'
)

# Menyimpan data yang telah diproses
df_rfm_final.to_csv(
    file_output,
    index=False
)

# Menampilkan beberapa data akhir
print("\n=== DATA SELESAI DISIMPAN ===")
print("Lokasi file:", file_output)
print("Jumlah baris:", df_rfm_final.shape[0])
print("Jumlah kolom:", df_rfm_final.shape[1])
print(df_rfm_final.head())

In [ ]:
# =========================
# 5. Cek Outlier untuk RFM
# =========================


# =========================
# Hitung Z-score & tandai outlier
# =========================
z_scores = stats.zscore(
    df_rfm_final[['Recency', 'Frequency', 'Monetary']]
)

outlier_condition = (
    np.abs(z_scores) > 3
).any(axis=1)


# =========================
# Scatter 3D Outlier
# =========================
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Normal points
ax.scatter(
    df_rfm_final.loc[~outlier_condition, 'Recency'],
    df_rfm_final.loc[~outlier_condition, 'Frequency'],
    df_rfm_final.loc[~outlier_condition, 'Monetary'],
    c='blue',
    alpha=0.6,
    label='Normal'
)

# Outlier points
ax.scatter(
    df_rfm_final.loc[outlier_condition, 'Recency'],
    df_rfm_final.loc[outlier_condition, 'Frequency'],
    df_rfm_final.loc[outlier_condition, 'Monetary'],
    c='red',
    marker='x',
    s=80,
    label='Outlier'
)

# Label sumbu
ax.set_xlabel('Recency (Hari)')
ax.set_ylabel('Frequency (Transaksi)')
ax.set_zlabel('Monetary (Rupiah)')

# Judul
ax.set_title(
    '3D Outlier Detection - Recency, Frequency, Monetary'
)

# Supaya sumbu Z tidak menggunakan scientific notation
formatter = ScalarFormatter(useOffset=False)
formatter.set_scientific(False)
ax.zaxis.set_major_formatter(formatter)

# Legend
ax.legend()

plt.show()


# =========================
# Tampilkan Outlier
# =========================
outliers = df_rfm_final[
    outlier_condition
].copy()

print("\n=== OUTLIERS ===")
print(outliers)

print(
    "\nJumlah outliers:",
    outliers.shape[0]
)


# =========================
# Simpan Data Outlier
# =========================
file_outlier = (
    '/content/drive/My Drive/TA/1 Tahun Lintang Bakso/'
    'data-outlier-rfm.csv'
)

outliers[
    [
        'Customer Phone',
        'Customer Name',
        'Recency',
        'Frequency',
        'Monetary'
    ]
].to_csv(
    file_outlier,
    index=False
)

print("\n=== DATA OUTLIER BERHASIL DISIMPAN ===")
print("Lokasi file:", file_outlier)
print("Jumlah outlier:", outliers.shape[0])


# =========================
# Hapus Outlier
# =========================
df_rfm_final_cleaned = df_rfm_final[
    ~outlier_condition
].copy()

print(
    "\nJumlah data sebelum menghapus outlier:",
    df_rfm_final.shape[0]
)

print(
    "Jumlah data setelah menghapus outlier:",
    df_rfm_final_cleaned.shape[0]
)

In [ ]:
# =========================
# 6. Penentuan Jumlah Klaster dengan Elbow
# =========================

# Ambil fitur RFM
X = df_rfm_final_cleaned[
    ['Recency', 'Frequency', 'Monetary']
]

sse = []
K_range = range(1, 11)

# Hitung SSE untuk setiap k
for k in K_range:
    kmeans = KMeans(
        n_clusters=k,
        init='k-means++',
        n_init=10,
        random_state=42
    )

    kmeans.fit(X)
    sse.append(kmeans.inertia_)


# =========================
# Tabel SSE + Penurunan SSE
# =========================

df_sse = pd.DataFrame({
    'k': list(K_range),
    'SSE': sse
})

# Konversi SSE ke triliun
df_sse['SSE'] = df_sse['SSE'] / 1e12

# Hitung penurunan SSE
df_sse['Penurunan_SSE'] = (
    df_sse['SSE'].shift(1) - df_sse['SSE']
)

# Bulatkan 2 angka desimal
df_sse['SSE'] = df_sse['SSE'].round(2)
df_sse['Penurunan_SSE'] = (
    df_sse['Penurunan_SSE'].round(2)
)

print("\n=== TABEL ELBOW METHOD ===")
print(df_sse)


# =========================
# Plot Elbow Method
# =========================

plt.figure(figsize=(8, 5))

plt.plot(
    df_sse['k'],
    df_sse['SSE'],
    marker='o'
)

plt.title('Elbow Method (SSE vs K)')
plt.xlabel('Jumlah Cluster (k)')
plt.ylabel('SSE (Triliun)')
plt.xticks(K_range)
plt.grid(True)
plt.show()

In [ ]:
# =========================
# 7. Segmentasi KMeans dan Visualisasi 3D dengan 2 Klaster
# =========================


# Pastikan DataFrame merupakan salinan independen
df_rfm_final_cleaned = df_rfm_final_cleaned.copy()

# Ambil fitur RFM
X = df_rfm_final_cleaned[
    ['Recency', 'Frequency', 'Monetary']
]

# Tentukan jumlah klaster berdasarkan hasil evaluasi, yaitu 2 klaster
kmeans = KMeans(
    n_clusters=2,
    init='k-means++',
    n_init=10,
    random_state=42
)

# Simpan hasil klaster ke DataFrame
df_rfm_final_cleaned.loc[:, 'Cluster'] = kmeans.fit_predict(X)

# Visualisasi klaster dalam 3D
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

scatter = ax.scatter(
    df_rfm_final_cleaned['Recency'],
    df_rfm_final_cleaned['Frequency'],
    df_rfm_final_cleaned['Monetary'],
    c=df_rfm_final_cleaned['Cluster'],
    cmap='viridis',
    s=50
)

# Label sumbu dan judul
ax.set_xlabel('Recency (Hari)')
ax.set_ylabel('Frequency (Transaksi)')
ax.set_zlabel('Monetary (Rupiah)')
ax.set_title('3D Visualization of Customer Segments (2 Clusters)')

# Supaya sumbu Monetary tidak memakai scientific notation
formatter = ScalarFormatter(useOffset=False)
formatter.set_scientific(False)
ax.zaxis.set_major_formatter(formatter)

# Color bar
fig.colorbar(scatter, ax=ax, label='Cluster')

# Tampilkan plot
plt.show()

In [ ]:
# =========================
# 8. Evaluasi Silhouette Coefficient
# =========================

X = df_rfm_final_cleaned[
    ['Recency', 'Frequency', 'Monetary']
]

sil_scores = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(
        n_clusters=k,
        init='k-means++',
        n_init=10,
        random_state=42
    )

    labels = kmeans.fit_predict(X)

    score = silhouette_score(X, labels)
    sil_scores.append(score)


# =========================
# Tabel Silhouette
# =========================

df_silhouette = pd.DataFrame({
    'k': list(K_range),
    'Silhouette_Coefficient': sil_scores
})

# Bulatkan 4 desimal
df_silhouette['Silhouette_Coefficient'] = (
    df_silhouette['Silhouette_Coefficient']
    .round(4)
)

print("\n=== TABEL SILHOUETTE COEFFICIENT ===")
print(df_silhouette)


# =========================
# Plot Silhouette Coefficient
# =========================

plt.figure(figsize=(8, 5))

plt.plot(
    df_silhouette['k'],
    df_silhouette['Silhouette_Coefficient'],
    marker='o'
)

plt.title('Silhouette Coefficient vs Jumlah Cluster (k)')
plt.xlabel('Jumlah Cluster (k)')
plt.ylabel('Silhouette Coefficient')
plt.xticks(list(K_range))
plt.grid(True)
plt.show()

In [ ]:
# # =========================
# # Davies-Bouldin Index
# # =========================

# # Ambil fitur RFM
# X = df_rfm_final_cleaned[['Recency', 'Frequency', 'Monetary']]

# dbi_scores = []
# K_range = range(2, 11)

# for k in K_range:
#     kmeans = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
#     labels = kmeans.fit_predict(X)

#     dbi = davies_bouldin_score(X, labels)
#     dbi_scores.append(dbi)

# # =========================
# # Plot Davies-Bouldin Index
# # =========================
# plt.figure(figsize=(8,5))
# plt.plot(K_range, dbi_scores, marker='o', color='red')
# plt.title('Davies-Bouldin Index vs Jumlah Cluster (k)')
# plt.xlabel('Jumlah Cluster (k)')
# plt.ylabel('Davies-Bouldin Index (lebih kecil lebih baik)')
# plt.xticks(K_range)
# plt.grid(True)
# plt.show()

In [ ]:
# =========================
# 9. Export Hasil Clustering ke Google Sheets
# =========================

# Install library jika belum tersedia
!pip install -q --upgrade gspread gspread-dataframe

import gspread
from gspread_dataframe import set_with_dataframe
from google.colab import auth
import google.auth


# =========================
# Auth Google Account
# =========================
auth.authenticate_user()

creds, _ = google.auth.default()
gc = gspread.authorize(creds)


# =========================
# Buat Google Sheet baru
# =========================
spreadsheet = gc.create(
    "Hasil Clustering RFM Lintang Bakso - Data 1 Tahun"
)

# Ambil worksheet pertama
worksheet = spreadsheet.get_worksheet(0)

# Ubah nama worksheet
worksheet.update_title("Hasil Clustering")


# =========================
# Siapkan data export
# =========================
df_export = df_rfm_final_cleaned[
    [
        'Customer Phone',
        'Customer Name',
        'Recency',
        'Frequency',
        'Monetary',
        'Cluster'
    ]
].copy()


# =========================
# Tulis ke Google Sheets
# =========================
set_with_dataframe(
    worksheet,
    df_export
)


# =========================
# Share agar bisa diakses Looker Studio
# =========================
spreadsheet.share(
    '',
    perm_type='anyone',
    role='reader'
)


# =========================
# Output link
# =========================
print("Export ke Google Sheets berhasil.")
print("Jumlah data:", df_export.shape[0])
print("Link:", spreadsheet.url)

In [ ]:
# =========================
# Distribusi RFM dan Ringkasan Cluster
# =========================


# =========================
# Jumlah Data
# =========================

jumlah_data = df_rfm_final_cleaned.shape[0]

print("\n=== JUMLAH DATA RFM ===")
print("Jumlah customer:", jumlah_data)


# =========================
# Distribusi RFM Setelah Outlier
# =========================

rfm_distribution = (
    df_rfm_final_cleaned[
        ['Recency', 'Frequency', 'Monetary']
    ]
    .describe()
)


print("\n=== DISTRIBUSI RFM ===")
print(rfm_distribution)


# Menampilkan mean RFM secara khusus
rfm_mean = (
    df_rfm_final_cleaned[
        ['Recency', 'Frequency', 'Monetary']
    ]
    .mean()
    .round(2)
)

print("\n=== MEAN RFM ===")
print(rfm_mean)


# =========================
# Ringkasan Persebaran Cluster
# =========================

cluster_summary = (
    df_rfm_final_cleaned
    .groupby('Cluster')
    .agg(
        Jumlah=('Customer Phone', 'count'),
        Avg_Recency=('Recency', 'mean'),
        Avg_Frequency=('Frequency', 'mean'),
        Avg_Monetary=('Monetary', 'mean')
    )
    .reset_index()
)


# Hitung persentase cluster
cluster_summary['Persentase (%)'] = (
    cluster_summary['Jumlah']
    / cluster_summary['Jumlah'].sum()
    * 100
).round(2)


# Pembulatan nilai rata-rata
cluster_summary[
    [
        'Avg_Recency',
        'Avg_Frequency',
        'Avg_Monetary'
    ]
] = cluster_summary[
    [
        'Avg_Recency',
        'Avg_Frequency',
        'Avg_Monetary'
    ]
].round(2)


# Urutan kolom
cluster_summary = cluster_summary[
    [
        'Cluster',
        'Jumlah',
        'Persentase (%)',
        'Avg_Recency',
        'Avg_Frequency',
        'Avg_Monetary'
    ]
]


print("\n=== RINGKASAN PERSEBARAN CLUSTER ===")
print(cluster_summary)